# Promover Seeds — Landing → Bronze → Silver

Promove as 6 tabelas de dimensão fixa (seeds) da Landing Zone para Bronze e Silver. Diferente das 11 tabelas de evento diário, seeds não têm sujeira intencional — só cast de tipo, sem UDFs de limpeza. Gravação por `overwrite`, não `MERGE` — o catálogo inteiro é recarregado a cada execução, não incremental.

Referências: ADR-001 (Landing Zone), ADR-013 (transformação, adaptada aqui para o caso mais simples).

In [0]:
# erp_produtos - Bronze

df_seed_bronze = spark.read.json("/Volumes/poc_pulse_observability/landing/raw/erp/_seed/erp_produtos.json")

from pyspark.sql.functions import col
df_seed_bronze = df_seed_bronze.select([col(c).cast("string").alias(c) for c in df_seed_bronze.columns])

df_seed_bronze.write.format("delta").mode("overwrite").saveAsTable("poc_pulse_observability.bronze.erp_produtos")

df_seed_bronze.printSchema()
df_seed_bronze.show()

In [0]:
# erp_produtos - Silver
from pyspark.sql.types import BooleanType, IntegerType

df_seed_silver = (
    df_seed_bronze
    .withColumn("exige_cadeia_fria", col("exige_cadeia_fria").cast(BooleanType()))
    .withColumn("validade_padrao_dias", col("validade_padrao_dias").cast(IntegerType()))
)

df_seed_silver.write.format("delta").mode("overwrite").saveAsTable("poc_pulse_observability.silver.erp_produtos")

df_seed_silver.printSchema()
df_seed_silver.show()

## Generalizando para os outros 5 seeds

Mesma decisão do ADR-013: função genérica + configuração declarativa, em vez de repetir o bloco Bronze+Silver 6 vezes.

In [0]:
# Promovendo os 6 seeds via função genérica
from src.transformacao.configuracao_seeds import CONFIGURACAO_SEEDS
from src.transformacao.promover_seed import promover_seed

for tabela, config in CONFIGURACAO_SEEDS.items():
    resultado = promover_seed(spark=spark, tabela=tabela, config=config)
    print(resultado)